# Week 1 - Day 2: TF-IDF + First ML Model

Today we convert cleaned text into numbers and train the first spam classifier.

## Goal
By the end of this notebook, we will:
- Convert text to TF-IDF features
- Train a Naive Bayes model
- Check accuracy
- Try custom predictions

## Step 1: Import libraries

In [ ]:
import os
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

## Step 2: Download NLTK resources locally

In [ ]:
LOCAL_NLTK_DIR = os.path.join(os.getcwd(), '.nltk_data')
os.makedirs(LOCAL_NLTK_DIR, exist_ok=True)
if LOCAL_NLTK_DIR not in nltk.data.path:
    nltk.data.path.insert(0, LOCAL_NLTK_DIR)

nltk.download('punkt', quiet=True, download_dir=LOCAL_NLTK_DIR)
nltk.download('punkt_tab', quiet=True, download_dir=LOCAL_NLTK_DIR)
nltk.download('stopwords', quiet=True, download_dir=LOCAL_NLTK_DIR)

print('NLTK resources ready')

NLTK resources ready


## Step 3: Load and prepare dataset

In [ ]:
df = pd.read_csv('spam.csv', encoding='latin-1')
df = df[['v1', 'v2']].copy()
df.columns = ['label', 'text']
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print('Total messages:', len(df))
print(df['label'].value_counts())
df.head()

Total messages: 5572
label
ham     4825
spam     747
Name: count, dtype: int64


,label,text,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## Step 4: Define text preprocessing

In [ ]:
def clean_text(text: str):
    text = str(text).lower()
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.isalnum()]
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

## Step 5: Apply cleaning and create TF-IDF input strings

In [ ]:
df['cleaned_text'] = df['text'].apply(clean_text)
df['cleaned_text_str'] = df['cleaned_text'].apply(lambda x: ' '.join(x))

df[['text', 'cleaned_text', 'cleaned_text_str']].head(3)

,text,cleaned_text,cleaned_text_str
0,"Go until jurong point, crazy.. Available only ...","[go, jurong, point, crazy, available, bugis, n...",go jurong point crazy available bugis n great ...
1,Ok lar... Joking wif u oni...,"[ok, lar, joking, wif, u, oni]",ok lar joking wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,"[free, entry, 2, wkly, comp, win, fa, cup, fin...",free entry 2 wkly comp win fa cup final tkts 2...


## Step 6: Convert text to numerical vectors with TF-IDF

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['cleaned_text_str']).toarray()
y = df['label_num']

print('Feature shape:', X.shape)

Feature shape: (5572, 3000)


## Step 7: Train first model (Naive Bayes)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = MultinomialNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print('Accuracy:', round(accuracy, 4))
print('\nClassification report:')
print(classification_report(y_test, y_pred, target_names=['ham', 'spam']))

Accuracy: 0.9767

Classification report:
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99       965
        spam       0.99      0.83      0.91       150

    accuracy                           0.98      1115
   macro avg       0.98      0.92      0.95      1115
weighted avg       0.98      0.98      0.98      1115



/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/NaveenkumarVasudevan/Downloads/Project/NLP/SMS Spam Collection Dataset/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow enco

## Step 8: Custom prediction function

In [ ]:
def predict_spam(text: str) -> str:
    cleaned = clean_text(text)
    cleaned_str = ' '.join(cleaned)
    vector = tfidf.transform([cleaned_str]).toarray()
    prediction = model.predict(vector)[0]
    return 'SPAM' if prediction == 1 else 'HAM'

print(predict_spam('Congratulations! You won a free iPhone'))
print(predict_spam('Hey, are we meeting today?'))

SPAM
HAM


## Day 2 Checklist
- [x] TF-IDF feature vectors created
- [x] Naive Bayes model trained
- [x] Accuracy evaluated
- [x] Custom predictions tested

### Next (Day 3)
Evaluate confusion matrix, precision/recall focus, and compare with Logistic Regression.